# Visualizations

This notebook reproduces the figures and token-level analyses for the GPT-2 POS circuit experiments in this directory.


## 1. Setup

In [ ]:
from pathlib import Path
import random
import re
import warnings

import numpy as np
import torch

# Make runs deterministic where possible.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ROOT = Path('.').resolve()
MODEL_NAME = 'gpt2'
C4_DATASET = 'datablations/c4-filter-small'
N_TEXT_SAMPLES = 500

REQUIRED_BASE_FILES = ['prct_diff.pth', 'pos_fv.pth']


def safe_torch_load(path):
    """Load torch artifacts safely across PyTorch versions."""
    path = str(path)
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except TypeError:
        return torch.load(path, map_location='cpu')


def require_file(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required artifact not found: {path}")
    return path


def discover_lm_cache_files(root=ROOT):
    return sorted(root.glob('lm*.pth'), key=lambda p: int(re.findall(r'\d+', p.name)[0]))


def lm_index_set(root=ROOT):
    return {int(re.findall(r'\d+', p.name)[0]) for p in discover_lm_cache_files(root)}


missing_base_files = [f for f in REQUIRED_BASE_FILES if not (ROOT / f).exists()]
if missing_base_files:
    warnings.warn('Missing required local artifacts: ' + ', '.join(missing_base_files))

LM_CACHE_FILES = discover_lm_cache_files(ROOT)
if not LM_CACHE_FILES:
    warnings.warn('No lm*.pth files found. Generate them with main.py before running token-level analyses.')


In [ ]:
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from IPython.display import HTML
from transformer_lens import HookedTransformer

warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
def visualize_tokens_responsive(tokens, scores, colormap='RdYlBu_r', 
                               container_width_percent=90, font_size=14):
    """
    Alternative version that uses CSS for responsive wrapping with working hover tooltips.
    
    Args:
        tokens: list of strings
        scores: list of numerical values  
        colormap: matplotlib colormap name
        container_width_percent: percentage of container width to use
        font_size: font size in pixels
    """
    # Normalize scores to [0, 1] for color mapping
    norm_scores = (np.array(scores) - np.min(scores)) / (np.max(scores) - np.min(scores))
    
    # Create colormap
    cmap = plt.get_cmap(colormap)
    
    # Generate unique container ID to avoid conflicts
    import random
    container_id = f"token_viz_{random.randint(1000, 9999)}"
    
    # Generate HTML tokens with data attributes for JavaScript
    html_tokens = []
    for i, (token, score, norm_score) in enumerate(zip(tokens, scores, norm_scores)):
        color = mcolors.rgb2hex(cmap(norm_score))
        html_tokens.append(
            f'<span class="token" style="background-color: {color};" '
            f'data-token="{token}" data-score="{score:.4f}" data-index="{i}">{token}</span>'
        )
    
    # Create color scale legend
    legend_colors = []
    n_legend = 10
    for i in range(n_legend):
        norm_val = i / (n_legend - 1)
        color = mcolors.rgb2hex(cmap(norm_val))
        score_val = np.min(scores) + norm_val * (np.max(scores) - np.min(scores))
        legend_colors.append(
            f'<span style="background-color: {color}; padding: 2px 6px; margin: 1px; '
            f'border-radius: 3px; font-size: 10px; display: inline-block;">{score_val:.2f}</span>'
        )
    
    html_content = f"""
    <style>
        .token-container {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            font-size: {font_size}px;
            width: {container_width_percent}%;
            padding: 15px;
            border: 1px solid #ddd;
            border-radius: 5px;
            background-color: #fafafa;
            line-height: 1.6;
            position: relative;
        }}
        .token {{
            padding: 2px 4px;
            margin: 1px;
            border-radius: 3px;
            white-space: nowrap;
            cursor: pointer;
            display: inline-block;
            position: relative;
        }}
        .token:hover {{
            opacity: 0.8;
            transform: scale(1.05);
            transition: all 0.2s ease;
            z-index: 100;
        }}
        .legend {{
            margin-bottom: 15px;
            padding-bottom: 10px;
            border-bottom: 1px solid #ddd;
        }}
        .legend-title {{
            font-weight: bold;
            margin-bottom: 5px;
        }}
        .legend-range {{
            font-size: 11px;
            color: #666;
            margin-top: 5px;
        }}
        .tooltip {{
            position: absolute;
            background-color: #333;
            color: white;
            padding: 8px 12px;
            border-radius: 4px;
            font-size: 12px;
            white-space: nowrap;
            z-index: 1000;
            pointer-events: none;
            opacity: 0;
            transition: opacity 0.3s;
            box-shadow: 0 2px 8px rgba(0,0,0,0.3);
        }}
        .tooltip.show {{
            opacity: 1;
        }}
        .tooltip::after {{
            content: '';
            position: absolute;
            top: 100%;
            left: 50%;
            margin-left: -5px;
            border-width: 5px;
            border-style: solid;
            border-color: #333 transparent transparent transparent;
        }}
    </style>
    
    <div id="{container_id}" class="token-container">
        <div class="legend">
            <div class="legend-title">Color Scale:</div>
            <div>{" ".join(legend_colors)}</div>
            <div class="legend-range">
                Range: {np.min(scores):.3f} to {np.max(scores):.3f} | 
                Tokens: {len(tokens)}
            </div>
        </div>
        
        <div class="text-content">
            {" ".join(html_tokens)}
        </div>
        
        <div class="tooltip" id="tooltip_{container_id}"></div>
    </div>
    
    <script>
    (function() {{
        const container = document.getElementById('{container_id}');
        const tooltip = document.getElementById('tooltip_{container_id}');
        const tokens = container.querySelectorAll('.token');
        
        tokens.forEach(token => {{
            token.addEventListener('mouseenter', function(e) {{
                const tokenText = this.getAttribute('data-token');
                const score = this.getAttribute('data-score');
                const index = this.getAttribute('data-index');
                
                tooltip.innerHTML = '<strong>' + tokenText + '</strong><br>Score: ' + score + '<br>Index: ' + index;
                tooltip.classList.add('show');
                
                // Position tooltip
                const rect = this.getBoundingClientRect();
                const containerRect = container.getBoundingClientRect();
                
                tooltip.style.left = (rect.left - containerRect.left + rect.width/2 - tooltip.offsetWidth/2) + 'px';
                tooltip.style.top = (rect.top - containerRect.top - tooltip.offsetHeight - 10) + 'px';
                
                // Ensure tooltip stays within container bounds
                const tooltipRect = tooltip.getBoundingClientRect();
                const containerBounds = container.getBoundingClientRect();
                
                if (tooltipRect.left < containerBounds.left) {{
                    tooltip.style.left = '5px';
                }}
                if (tooltipRect.right > containerBounds.right) {{
                    tooltip.style.left = (containerBounds.width - tooltipRect.width - 5) + 'px';
                }}
                if (tooltipRect.top < containerBounds.top) {{
                    tooltip.style.top = (rect.bottom - containerRect.top + 5) + 'px';
                }}
            }});
            
            token.addEventListener('mouseleave', function() {{
                tooltip.classList.remove('show');
            }});
        }});
    }})();
    </script>
    """
    
    return HTML(html_content)

## 2. Figure 1: POS Circuit / Important Attention Heads

In [ ]:
prct_diff_path = require_file(ROOT / "prct_diff.pth")
a = safe_torch_load(prct_diff_path)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(a, cmap=sns.cubehelix_palette(as_cmap=True), square=True, linewidth=0.5, ax=ax)
ax.set_title("GPT-2 POS Circuit: Head Importance")
ax.set_ylabel("Layer Index")
ax.set_xlabel("Attention Head Index")
fig.tight_layout()
fig.savefig("importance.pdf", bbox_inches="tight")

## 3. Token-Level Function Vector Similarity

In [ ]:
from datasets import load_dataset

ds = load_dataset(C4_DATASET)
text = ds["train"]["text"][:N_TEXT_SAMPLES]
print(f"Loaded {len(text)} text samples from {C4_DATASET}.")

In [ ]:
if len(text) == 0:
    raise ValueError("Dataset returned no text samples. Check dataset availability.")

In [ ]:
gpt2 = HookedTransformer.from_pretrained(MODEL_NAME)

In [ ]:
fv_path = require_file(ROOT / "pos_fv.pth")
fv = safe_torch_load(fv_path)

In [ ]:
def co_sim(a, b):
    return (a.dot(b) / (max(a.norm(p=2) * b.norm(p=2), 1e-8)))

def angular(a, b):
    return 1 - torch.arccos(co_sim(a,b)) / torch.pi

def get_sim(cache, sim_fn):
    n_layers = gpt2.cfg.n_layers
    layers = []
    
    for i in range(n_layers):
        layers.append(cache[f"blocks.{i}.attn.hook_z"].mean(dim=-2)[0])
    return np.array([[sim_fn(fv, v).item() for v in layers[i]] for i in range(n_layers)]).mean(axis=0)

In [ ]:
## MEAN TOKEN REPRESENTATION SIMILARITY BETWEEN LAST HIDDEN STATE AND FUNCTION VECTOR
MEAN = 0.016796189400105643

In [ ]:
def ami(lall):
    return np.where(lall > MEAN, 1, 0)

In [ ]:
def ambiguity(p):
    return 1 - np.abs(2 * p - 1)

In [ ]:
def stats(t):
    if t >= len(text):
        raise IndexError(f"Sample index {t} out of range for {len(text)} texts.")
    lm_path = require_file(ROOT / f"lm{t}.pth")
    cache = safe_torch_load(lm_path)
    lall = get_sim(cache, co_sim)
    toks = gpt2.to_str_tokens(text[t])
    return toks, ami(lall)

In [ ]:
def visualize(t):
    if t >= len(text):
        raise IndexError(f"Sample index {t} out of range for {len(text)} texts.")
    lm_path = require_file(ROOT / f"lm{t}.pth")
    cache = safe_torch_load(lm_path)
    lall = get_sim(cache, co_sim)
    toks = gpt2.to_str_tokens(text[t])
    return visualize_tokens_responsive(toks, ami(lall))

In [ ]:
visualize(12)

In [ ]:
LM_CACHE_FILES = discover_lm_cache_files(ROOT)
nums = [int(re.findall(r'\d+', p.name)[0]) for p in LM_CACHE_FILES]

if not nums:
    raise FileNotFoundError("No lm*.pth files found. Run main.py to generate cache files.")

print(f"Found {len(nums)} LM cache files.")

In [ ]:
# Aggregate token-level scores across all available lm*.pth files.

total_stats = [stats(v) for v in nums]

d = {}
all_scores = []

for words, scores in total_stats:
    for i in range(len(words)):
        token = words[i].strip()
        if token == "":
            continue
        d.setdefault(token, []).append(scores[i])
        all_scores.append(scores[i])

print(f"Aggregated {len(all_scores)} token scores across {len(d)} unique tokens.")


In [ ]:
avgs = dict()
for k, v in d.items():
    avgs[k] = ambiguity(np.mean(v))

In [ ]:
print(f"Std(all_scores): {np.std(all_scores):.4f}")
print(f"Mean(all_scores): {np.mean(all_scores):.4f}")
print(f"N(all_scores): {len(all_scores)}")

In [ ]:
ranked = sorted(avgs.items(), key=lambda x: x[1], reverse=True)

print("Top 10 tokens by ambiguity score:")
for k, v in ranked[:10]:
    print(f"{k}: {v:.4f}")

print("\nBottom 10 tokens by ambiguity score:")
for k, v in ranked[-10:]:
    print(f"{k}: {v:.4f}")

In [ ]:
def summarize_token_group(name, tokens):
    vals = []
    for tok in tokens:
        vals.extend(d.get(tok, []))
    if len(vals) == 0:
        print(f"{name}: no matched tokens")
        return None
    print(f"{name}: n={len(vals)}, std={np.std(vals):.4f}, mean={np.mean(vals):.4f}")
    return vals

# Example linguistic groups used in the paper narrative.
punctuation_tokens = [".", ",", ";", ":", "-", "\"", ")", "(", "!", "?", "?\""]
article_tokens = ["a", "an", "the"]
preposition_tokens = ["to", "on", "in", "at", "for", "with", "about", "as", "into", "like", "through", "after", "over", "between", "out", "against", "during", "without", "before", "under"]
verb_tokens = ["buy", "get", "go", "make", "see", "know", "think", "take", "come", "want", "look", "use", "find", "give", "tell", "work", "call", "try", "ask", "need", "feel", "be", "have", "do", "say", "would", "could", "should", "might", "will", "shall"]

summarize_token_group("Punctuation", punctuation_tokens)
summarize_token_group("Articles", article_tokens)
summarize_token_group("Prepositions", preposition_tokens)
summarize_token_group("Verbs", verb_tokens)

## 4. Appendix: Optional Sanity Checks and Extra Examples

The cells below are optional and can be run independently after completing Sections 1-3.

In [ ]:
example_indices = [12, 24, 25, 59]
available_indices = lm_index_set(ROOT)
print(f"Available cache indices: {len(available_indices)} files")
print("Examples with available caches:", [i for i in example_indices if i in available_indices])

In [ ]:
# Optional: visualize one sample from the available cache files.
example_t = next(iter(sorted(available_indices)))
visualize(example_t)

In [ ]:
# Optional sanity check: inspect all unique tokens discovered.
sorted(list(d.keys()))[:100]

In [ ]:
# Optional token lookup example.
d.get("buy", [])[:20]

In [ ]:
# Optional: render specific examples if cache files exist.
for t in [12, 24, 25, 59]:
    if t in available_indices:
        print(f"Rendering t={t}")
        display(visualize(t))